In [ ]:
# tools/polygon_load_transfer.py
from __future__ import annotations

from collections import defaultdict
from copy import deepcopy
import math
from typing import Any

import numpy as np

from data.enums import DistType, LoadDirection
from domain.definition import AreaUniformLoad, FrameDistributedLoad
from domain.polygon import Polygon

_EPS = 1e-9

_POLYGON_LOCAL_TO_AXIS = {
    LoadDirection.LOCAL_1: 0,
    LoadDirection.LOCAL_2: 1,
    LoadDirection.LOCAL_3: 2,
}


def build_polygon_frame_loads(
    math_model,
    polygon,
    loads=None,
    *,
    analysis_data=None,
    dist_type=None,
    step: float = 200.0,
    n_bins: int = 12,
    zone_fields: list[tuple[list[tuple[float, float, float]], AreaUniformLoad]] | None = None,
) -> dict[int, dict[str, FrameDistributedLoad]]:
    if math_model is None or polygon is None:
        raise ValueError("MathModel ve polygon gereklidir.")
    if len(getattr(polygon, "nodes", ())) < 3:
        raise ValueError("Yük aktarımı için polygon en az üç node içermelidir.")
    if analysis_data is None:
        raise ValueError("PolygonAnalysisData gerekli.")

    selected_loads = list(loads if loads is not None else analysis_data.assigned_loads)
    if not selected_loads and not zone_fields:
        raise ValueError("Polygon üzerinde aktarılacak alan yükü yok.")

    associated_ids = {int(frame_id) for frame_id in analysis_data.associated_frame_ids}
    if not associated_ids:
        raise ValueError("Önce polygon için aktarım hedefi frame'leri seçin.")

    frames = [
        frame for frame in math_model.frames.values()
        if int(frame.unique_id) in associated_ids
    ]
    if len(frames) != len(associated_ids):
        found = {int(frame.unique_id) for frame in frames}
        missing = sorted(associated_ids - found)
        raise ValueError(f"İlişkili frame bulunamadı: {missing}")

    u, v, normal, origin, polygon_2d = _polygon_coordinates(polygon)
    area = _polygon_area(polygon_2d)
    if area <= _EPS:
        raise ValueError("Polygon alanı sıfır veya geçersiz.")

    frame_data = []
    for frame in frames:
        data = _project_frame(frame, origin, u, v, normal, polygon_2d)
    
        if data is None:
            print(f"ELENDİ: Frame {frame.unique_id}")
        else:
            print(f"KABUL: Frame {frame.unique_id}")
            frame_data.append(data)
    if not frame_data:
        raise ValueError("İlişkili frame'lerden hiçbiri polygon düzlemiyle kesişmiyor.")

    distribution = DistType.from_input(
        dist_type if dist_type is not None else analysis_data.transfer_dist_type
    )

    result: dict[int, dict[str, FrameDistributedLoad]] = defaultdict(dict)

    if selected_loads:
        q_profiles = None
        if distribution == DistType.TWO_WAY:
            q_profiles = _tributary_q_profiles_grid(
                polygon_2d,
                frame_data,
                step=step,
                n_bins=n_bins,
                distribution=distribution,
            )
        tributary_areas = None if q_profiles is not None else _tributary_areas(
            polygon_2d, frame_data, distribution
        )

        for load in selected_loads:
            if not isinstance(load, AreaUniformLoad):
                raise TypeError("AreaUniformLoad bekleniyor.")
            if not load.pattern_name:
                raise ValueError("Pattern adı boş olamaz.")
            _accumulate_uniform_load(
                result=result,
                load=load,
                frame_data=frame_data,
                polygon_axes=(u, v, normal),
                distribution=distribution,
                q_profiles=q_profiles,
                tributary_areas=tributary_areas,
            )
    print("result:", result)
    print("FRAME DATA:")
    for fd in frame_data:
        print(fd['frame'].unique_id)
    if zone_fields:
        zone_layers_by_load: dict[tuple[str, LoadDirection, str], list[tuple[np.ndarray, float]]] = defaultdict(list)
        for coords_3d, z_load in zone_fields:
            pts_3d = np.asarray(coords_3d, dtype=float)
            if pts_3d.ndim != 2 or pts_3d.shape[0] < 3:
                continue
            delta = pts_3d - origin
            pts_2d = np.column_stack((delta @ u, delta @ v))
            key = (z_load.pattern_name, z_load.direction, z_load.coordinate_system)
            zone_layers_by_load[key].append((pts_2d, float(z_load.value)))

        for (pattern_name, direction, coord_sys_name), layers in zone_layers_by_load.items():
            q_profiles = _tributary_q_profiles_grid(
                polygon_2d,
                frame_data,
                step=step,
                n_bins=n_bins,
                distribution=distribution,
                zone_layers=layers,
            )
            if not q_profiles:
                continue
            probe_load = AreaUniformLoad(
                pattern_name=pattern_name,
                direction=direction,
                value=1.0,
                coordinate_system=coord_sys_name,
            )
            
            
                
            _accumulate_uniform_load(
                result=result,
                load=probe_load,
                frame_data=frame_data,
                polygon_axes=(u, v, normal),
                distribution=distribution,
                q_profiles=q_profiles,
                tributary_areas=None,
            )

        print("RESULT KEYS:", list(result.keys()))

        for frame_id, patterns in result.items():
            print(frame_id)

    return dict(result)

In [ ]:
def _accumulate_uniform_load(
    *,
    result: dict[int, dict[str, FrameDistributedLoad]],
    load: AreaUniformLoad,
    frame_data: list[dict],
    polygon_axes: tuple[np.ndarray, np.ndarray, np.ndarray],
    distribution: DistType,
    q_profiles: dict[int, list[tuple[float, float, float, float]]] | None,
    tributary_areas: dict[int, float] | None,
) -> None:
    for item in frame_data:
        frame = item["frame"]
        frame_id = int(frame.unique_id)
        effective_length = item["effective_length"]
        if effective_length < _EPS:
            continue
        resolved_directions = _resolve_frame_load_directions(frame, load, polygon_axes)
        print(f"frame_id: {frame_id}, resolved: {resolved_directions}")
        for (coord_sys, load_dir), direction_sign in resolved_directions:
            
            
    
            load_set = result[frame_id].get(load.pattern_name)
            if load_set is None:
                load_set = FrameDistributedLoad(
                    pattern_name=load.pattern_name,
                    coord_sys=coord_sys,
                    load_dir=load_dir,
                    mode="REPLACE",
                    direction=load.direction,
                    coordinate_system=load.coordinate_system,
                    dist_type="two_way" if distribution == DistType.TWO_WAY else "one_way",
                )
                result[frame_id][load.pattern_name] = load_set
            
    
            if q_profiles is not None:
                segments = q_profiles.get(frame_id)
                print(
            f"FRAME={frame_id}, "
            f"PROFILE={len(segments) if segments else 0}, "
            f"LENGTH={effective_length}"
        )
                if not segments:
                    continue
                for x1_rel, x2_rel, q1, q2 in segments:
                    q1_final = direction_sign * q1 * float(load.value)
                    q2_final = direction_sign * q2 * float(load.value)
                    if abs(q1_final) <= _EPS and abs(q2_final) <= _EPS:
                        continue
                    load_set.add_segment(x1_rel, x2_rel, q1_final, q2_final)
            elif tributary_areas is not None:
                ta = tributary_areas.get(frame_id, 0.0)
                q_line = direction_sign * float(load.value) * ta / effective_length
                if abs(q_line) <= _EPS:
                    continue
                for start, end in item["intervals"]:
                    load_set.add_segment(start, end, q_line, q_line)

In [ ]:
def merge_frame_distributed_loads(
    math_model,
    staged: dict[int, dict[str, FrameDistributedLoad]],
    mode: str,
) -> list[int]:
    touched: list[int] = []
    for frame_id, pattern_loads in staged.items():
        frame = math_model.frames.get(int(frame_id))
        if frame is None:
            continue
        changed = False
        for pattern_name, incoming in pattern_loads.items():
            if mode == "delete":
                if pattern_name in frame.dist_load_sets:
                    del frame.dist_load_sets[pattern_name]
                    changed = True
            elif mode == "replace":
                frame.dist_load_sets[pattern_name] = deepcopy(incoming)
                changed = True
            elif mode == "add":
                existing = frame.dist_load_sets.get(pattern_name)
                if existing is None:
                    frame.dist_load_sets[pattern_name] = deepcopy(incoming)
                    changed = True
                elif (existing.coord_sys, existing.load_dir) == (incoming.coord_sys, incoming.load_dir):
                    for seg in incoming.segments:
                        existing.add_segment(seg.x1, seg.x2, seg.q1, seg.q2)
                    changed = True
        if changed:
            frame.mark_dirty()
            touched.append(int(frame_id))
    return touched


def _polygon_coordinates(polygon):
    nodes = list(polygon.nodes)
    points = np.asarray([(node.x, node.y, node.z) for node in nodes], dtype=float)
    origin = points.mean(axis=0)
    normal = np.asarray(Polygon.calculate_normal(nodes), dtype=float)
    normal_length = float(np.linalg.norm(normal))
    if not np.isfinite(normal).all() or normal_length <= _EPS:
        raise ValueError("Polygon düzlemi hesaplanamadı.")
    normal /= normal_length

    first_edge = points[1] - points[0]
    first_edge -= normal * float(np.dot(first_edge, normal))
    edge_length = float(np.linalg.norm(first_edge))
    if edge_length <= _EPS:
        raise ValueError("Polygonun ilk kenarı geçersiz.")
    u = first_edge / edge_length
    v = np.cross(normal, u)
    v /= float(np.linalg.norm(v))

    delta = points - origin
    distances = np.abs(delta @ normal)
    polygon_scale = float(np.linalg.norm(points.max(axis=0) - points.min(axis=0)))
    tol = max(polygon_scale, 1.0) * 1e-6
    if float(distances.max(initial=0.0)) > tol:
        if float(distances.max()) > polygon_scale * 0.05:
            raise ValueError("Polygon düzlemden çok sapıyor.")
        points = points - np.outer(delta @ normal, normal)

    polygon_2d = np.column_stack(((points - origin) @ u, (points - origin) @ v))
    return u, v, normal, origin, polygon_2d


def _project_frame(frame, origin, u, v, normal, polygon_2d):
    if not getattr(frame, "node_i", None) or not getattr(frame, "node_j", None):
        return None
    points = np.asarray(
        [
            (frame.node_i.x, frame.node_i.y, frame.node_i.z),
            (frame.node_j.x, frame.node_j.y, frame.node_j.z),
        ],
        dtype=float,
    )
    if not np.isfinite(points).all():
        return None
    delta = points - origin
    frame_len = float(np.linalg.norm(points[1] - points[0]))
    polygon_scale = float(np.linalg.norm(polygon_2d.max(axis=0) - polygon_2d.min(axis=0)))
    tol = max(frame_len, polygon_scale, 1.0) * 1e-5
    distance = float(np.max(np.abs(delta @ normal)))
    if distance > tol:
        return None
    if distance > tol * 0.01:
        points = points - np.outer(delta @ normal, normal)
        delta = points - origin

    points_2d = np.column_stack((delta @ u, delta @ v))
    intervals = _segment_polygon_intervals(points_2d[0], points_2d[1], polygon_2d)
    if not intervals:
        return None
    effective_fraction = sum(end - start for start, end in intervals)
    effective_length = frame_len * effective_fraction
    if frame_len <= _EPS or effective_length <= _EPS:
        return None

    midpoint = sum(
        (end - start) * (points_2d[0] + (points_2d[1] - points_2d[0]) * ((start + end) / 2))
        for start, end in intervals
    ) / effective_fraction
    return {
        "frame": frame,
        "pi_2d": (float(points_2d[0][0]), float(points_2d[0][1])),
        "pj_2d": (float(points_2d[1][0]), float(points_2d[1][1])),
        "intervals": intervals,
        "effective_length": effective_length,
        "full_length": frame_len,
        "midpoint": midpoint,
    }


def _segment_polygon_intervals(start, end, polygon):
    direction = end - start
    if float(np.linalg.norm(direction)) <= _EPS:
        return []
    breaks = [0.0, 1.0]
    for index, edge_start in enumerate(polygon):
        edge_end = polygon[(index + 1) % len(polygon)]
        edge = edge_end - edge_start
        denominator = _cross_2d(direction, edge)
        if abs(denominator) <= _EPS:
            continue
        offset = edge_start - start
        t = _cross_2d(offset, edge) / denominator
        s = _cross_2d(offset, direction) / denominator
        if -_EPS <= t <= 1.0 + _EPS and -_EPS <= s <= 1.0 + _EPS:
            breaks.append(min(1.0, max(0.0, t)))
    breaks = sorted(set(round(value, 12) for value in breaks))
    intervals = []
    for t0, t1 in zip(breaks, breaks[1:]):
        if t1 - t0 <= _EPS:
            continue
        if _point_in_polygon(start + direction * ((t0 + t1) / 2), polygon):
            intervals.append((t0, t1))
    return intervals


def _point_in_polygon(point, polygon) -> bool:
    x, y = point
    inside = False
    for index, first in enumerate(polygon):
        second = polygon[(index + 1) % len(polygon)]
        edge = second - first
        offset = np.asarray((x, y), dtype=float) - first
        scale = max(float(np.linalg.norm(edge)), 1.0)
        if (
            abs(_cross_2d(edge, offset)) <= _EPS * scale
            and -_EPS <= float(np.dot(offset, edge)) <= float(np.dot(edge, edge)) + _EPS
        ):
            return True
        if (first[1] > y) != (second[1] > y):
            crossing_x = (
                (second[0] - first[0]) * (y - first[1]) / (second[1] - first[1]) + first[0]
            )
            if x < crossing_x:
                inside = not inside
    return inside


def _tributary_areas(polygon, frames, distribution):
    areas = defaultdict(float)
    axis = 1 if distribution in (DistType.ONE_WAY, DistType.ONE_WAY_U1) else 0
    groups: list[list[dict[str, Any]]] = []
    for item in sorted(frames, key=lambda value: value["midpoint"][axis]):
        coordinate = float(item["midpoint"][axis])
        if groups and abs(coordinate - float(groups[-1][0]["midpoint"][axis])) <= _EPS:
            groups[-1].append(item)
        else:
            groups.append([item])

    centers = [float(group[0]["midpoint"][axis]) for group in groups]
    for index, group in enumerate(groups):
        lower = (centers[index - 1] + centers[index]) / 2.0 if index > 0 else None
        upper = (centers[index] + centers[index + 1]) / 2.0 if index + 1 < len(groups) else None
        cell = _clip_axis(polygon, axis, lower, upper)
        cell_area = _polygon_area(cell)
        total_length = sum(item["effective_length"] for item in group)
        if total_length <= _EPS:
            continue
        for item in group:
            frame_id = int(item["frame"].unique_id)
            areas[frame_id] += cell_area * item["effective_length"] / total_length
    return dict(areas)


def _tributary_q_profiles_grid(
    polygon_2d: np.ndarray,
    frame_data: list[dict],
    *,
    step: float = 200.0,
    n_bins: int = 12,
    distribution: DistType = DistType.TWO_WAY,
    zone_layers: list[tuple[np.ndarray, float]] | None = None,
) -> dict[int, list[tuple[float, float, float, float]]]:
    min_u, min_v = polygon_2d.min(axis=0)
    max_u, max_v = polygon_2d.max(axis=0)
    if step <= 0:
        step = 200.0

    dA = step * step
    nu = max(1, int(math.ceil((max_u - min_u) / step)))
    nv = max(1, int(math.ceil((max_v - min_v) / step)))
    bins: dict[int, dict[int, float]] = {int(fd["frame"].unique_id): {} for fd in frame_data}
    one_way_axis = 1 if distribution in (DistType.ONE_WAY, DistType.ONE_WAY_U1) else 0

    for iu in range(nu):
        for iv in range(nv):
            cu = float(min_u + (iu + 0.5) * step)
            cv = float(min_v + (iv + 0.5) * step)
            if not _point_in_polygon((cu, cv), polygon_2d):
                continue

            if zone_layers is not None:
                cell_pressure = 0.0
                for zone_poly, z_val in zone_layers:
                    if _point_in_polygon((cu, cv), zone_poly):
                        cell_pressure += z_val
                if abs(cell_pressure) <= _EPS:
                    continue
            else:
                cell_pressure = 1.0

            q_cell = cell_pressure * dA
            best_fd = None
            best_dist = float("inf")
            best_t = 0.0

            for fd in frame_data:
                t, dist = _point_segment_param((cu, cv), fd["pi_2d"], fd["pj_2d"])
                if distribution != DistType.TWO_WAY:
                    px = fd["pi_2d"][0] + (fd["pj_2d"][0] - fd["pi_2d"][0]) * t
                    py = fd["pi_2d"][1] + (fd["pj_2d"][1] - fd["pi_2d"][1]) * t
                    dist = abs((cv - py) if one_way_axis == 1 else (cu - px))
                if dist < best_dist:
                    best_dist = dist
                    best_fd = fd
                    best_t = t

            if best_fd is None:
                continue

            eff_len = float(best_fd["effective_length"])
            full_len = float(best_fd["full_length"])
            intervals = best_fd["intervals"]
            if eff_len <= _EPS or not intervals:
                continue

            x_eff = _t_to_effective_x(best_t, intervals, full_len)
            bin_size = eff_len / max(1, n_bins)
            bin_idx = min(n_bins - 1, max(0, int(x_eff / bin_size)))
            fid = int(best_fd["frame"].unique_id)
            bins[fid][bin_idx] = bins[fid].get(bin_idx, 0.0) + q_cell

    result: dict[int, list[tuple[float, float, float, float]]] = {}
    for fd in frame_data:
        fid = int(fd["frame"].unique_id)
        bin_dict = bins.get(fid, {})
        eff_len = float(fd["effective_length"])
        if eff_len <= _EPS or not bin_dict:
            continue

        bin_size = eff_len / n_bins
        q_by_bin = [0.0] * n_bins
        for bin_idx, force in bin_dict.items():
            q_by_bin[bin_idx] = force / bin_size

        if max(abs(v) for v in q_by_bin) <= _EPS:
            continue

        segments: list[tuple[float, float, float, float]] = []
        for i in range(n_bins):
            q_mid = q_by_bin[i]
            q_left = q_by_bin[i - 1] if i > 0 else q_mid
            q_right = q_by_bin[i + 1] if i < n_bins - 1 else q_mid
            x1_rel = i / n_bins
            x2_rel = (i + 1) / n_bins
            q1 = (q_left + q_mid) * 0.5
            q2 = (q_mid + q_right) * 0.5
            if abs(q1) <= _EPS and abs(q2) <= _EPS:
                continue
            segments.append((x1_rel, x2_rel, q1, q2))
        if segments:
            result[fid] = segments

    return result


def _t_to_effective_x(t_full: float, intervals: list[tuple[float, float]], full_len: float) -> float:
    x_eff = 0.0
    for t0, t1 in intervals:
        if t_full <= t0:
            break
        if t_full >= t1:
            x_eff += (t1 - t0) * full_len
        else:
            x_eff += (t_full - t0) * full_len
            break
    return x_eff


def _point_segment_param(pt, a, b) -> tuple[float, float]:
    px, py = pt
    ax, ay = a
    bx, by = b
    dx = bx - ax
    dy = by - ay
    L2 = dx * dx + dy * dy
    if L2 < 1e-12:
        return 0.0, math.hypot(px - ax, py - ay)
    t = max(0.0, min(1.0, ((px - ax) * dx + (py - ay) * dy) / L2))
    return t, math.hypot(px - (ax + dx * t), py - (ay + dy * t))


def _clip_axis(polygon, axis, lower, upper):
    clipped = list(polygon)
    if lower is not None:
        clipped = _clip_half_plane(clipped, -np.eye(2)[axis], -float(lower))
    if upper is not None:
        clipped = _clip_half_plane(clipped, np.eye(2)[axis], float(upper))
    return clipped


def _clip_half_plane(polygon, normal, boundary):
    if len(polygon) == 0:
        return []
    result = []
    previous = polygon[-1]
    previous_inside = float(np.dot(normal, previous)) <= boundary + _EPS
    for current in polygon:
        current_inside = float(np.dot(normal, current)) <= boundary + _EPS
        if current_inside != previous_inside:
            direction = current - previous
            denominator = float(np.dot(normal, direction))
            if abs(denominator) > _EPS:
                t = (boundary - float(np.dot(normal, previous))) / denominator
                result.append(previous + direction * t)
        if current_inside:
            result.append(current)
        previous = current
        previous_inside = current_inside
    return result


def _polygon_area(polygon):
    if len(polygon) < 3:
        return 0.0
    points = np.asarray(polygon, dtype=float)
    return abs(
        float(
            np.dot(points[:, 0], np.roll(points[:, 1], 1))
            - np.dot(points[:, 1], np.roll(points[:, 0], 1))
        )
    ) / 2.0


def _cross_2d(first, second):
    return float(first[0] * second[1] - first[1] * second[0])


def _frame_local_axes(frame) -> np.ndarray:
    return np.asarray(frame.get_transformation_matrix(), dtype=float)


def _match_local_axis(frame, polygon_axes, load_direction, cos_tol=0.99):
    poly_idx = _POLYGON_LOCAL_TO_AXIS[load_direction]
    poly_axis = np.asarray(polygon_axes[poly_idx], dtype=float)
    axis_len = float(np.linalg.norm(poly_axis))
    if axis_len <= _EPS:
        return None, 1.0
    poly_axis /= axis_len

    frame_axes = _frame_local_axes(frame)
    dots = frame_axes.T @ poly_axis
    abs_dots = np.abs(dots)
    best_idx = int(np.argmax(abs_dots))
    if abs_dots[best_idx] < cos_tol:
        return None, 1.0
    sign = 1.0 if dots[best_idx] >= 0.0 else -1.0
    return best_idx + 1, sign


def _resolve_frame_load_directions(frame, load, polygon_axes):
    direction = load.direction

    if direction == LoadDirection.GRAVITY:
        return [(("Global", "7"), 1.0)]

    if direction == LoadDirection.GLOBAL_X:
        return [(("Global", "4"), 1.0)]

    if direction == LoadDirection.GLOBAL_Y:
        return [(("Global", "5"), 1.0)]

    if direction == LoadDirection.GLOBAL_Z:
        return [(("Global", "6"), 1.0)]

    poly_idx = _POLYGON_LOCAL_TO_AXIS[direction]
    poly_axis = np.asarray(polygon_axes[poly_idx], dtype=float)
    poly_axis /= np.linalg.norm(poly_axis)

    frame_axes = _frame_local_axes(frame)
    components = frame_axes.T @ poly_axis
    print(
        f"Frame {frame.unique_id}: "
        f"L1={components[0]:.6f}, "
        f"L2={components[1]:.6f}, "
        f"L3={components[2]:.6f}"
    )
    return [
        (("Local", str(i + 1)), float(components[i]))
        for i in (1, 2)
        if abs(components[i]) > _EPS
    ]

In [ ]:
res_=build_polygon_frame_loads(
            proj.math_model,
            poly,
            loads=loads,
            analysis_data=ad,
            dist_type=DistType.from_input(ad.transfer_dist_type),
            step=step,
            n_bins=n_bins,
            zone_fields=zone_fields,
        )

In [ ]:
# core/units.py

from __future__ import annotations

import logging
from collections.abc import Callable
from enum import Enum
from typing import Any

logger = logging.getLogger(__name__)


# ============================================================================
# FİZİKSEL SABİTLER (SI)
# ============================================================================
G_M_S2 = 9.80665
G_MM_S2 = 9806.65
KG_PER_TON = 1000.0
MEGAGRAM = 1000.0

DENSITY_STEEL = 7.85e-9
DENSITY_CONCRETE = 2.50e-9
DENSITY_ALUMINUM = 2.70e-9
DENSITY_TIMBER = 0.60e-9


# ============================================================================
# BİRİM TABLOLARI
# ============================================================================
FORCE_TO_N: dict[str, float] = {
    "N": 1.0, "kN": 1e3, "kgf": 9.80665, "tonf": 9806.65,
    "lb": 4.4482216152605, "kip": 4448.2216152605,
}

LENGTH_TO_MM: dict[str, float] = {
    "mm": 1.0, "cm": 10.0, "m": 1000.0, "in": 25.4, "ft": 304.8,
}

MASS_TO_KG: dict[str, float] = {
    "kg": 1.0, "t": 1000.0, "tonf": 1000.0, "ton": 1000.0,
    "g": 1e-3, "lb": 0.45359237, "kip": 453.59237,
}

# Mutlak sıcaklık: °C = (v - b) * a
TEMP_FACTOR: dict[str, tuple[float, float]] = {
    "C": (1.0, 0.0),
    "F": (5.0 / 9.0, 32.0),
}


# ============================================================================
# BOYUTLAR
# ============================================================================
class Dimension(Enum):
    LENGTH = "L"
    FORCE = "F"
    STRESS = "F/L²"
    AREA = "L²"
    I3 = "L³"
    I4 = "L⁴"
    I6 = "L6"
    MOMENT = "F·L"
    MASS = "M"
    MASS_INERTIA = "M·L²"
    MASS_DENSITY = "M/L³"
    WEIGHT_DENSITY = "F/L³"
    LINE_LOAD = "F/L"
    AREA_LOAD = "F/L²"
    ACCEL = "L/T²"
    TEMP = "Θ"
    THERMAL_EXP = "1/Θ"


class UnitSystem(Enum):
    N_MM = "N, mm, C"
    N_M = "N, m, C"
    KN_MM = "kN, mm, C"
    KN_M = "kN, m, C"
    KGF_MM = "kgf, mm, C"
    KGF_M = "kgf, m, C"
    KGF_CM = "kgf, cm, C"
    TONF_MM = "tonf, mm, C"
    TONF_M = "tonf, m, C"
    LB_IN = "lb, in, F"
    KIP_IN = "kip, in, F"
    KIP_FT = "kip, ft, F"

    @classmethod
    def from_sap(cls, s: str | None) -> UnitSystem:
        """SAP2000 CurrUnits string → UnitSystem.

        v25:  "Tonf, m, C"
        v10:  "Ton. m. C"   (nokta + boşluk ayraç; "Ton" = "tonf")
        """
        raw = (s or "N, mm, C").strip()
        if not raw:
            return cls.N_MM

        low = raw.lower()
        for us in cls:
            if us.value.lower() == low:
                return us

        normalized = cls._normalize_v10(raw)
        low_n = normalized.lower()
        for us in cls:
            if us.value.lower() == low_n:
                return us

        parts = [p.strip().lower() for p in normalized.split(",") if p.strip()]
        if len(parts) >= 2:
            force = parts[0]
            length = parts[1]
            if force == "ton":
                force = "tonf"
            for us in cls:
                if us.force_unit.lower() == force and us.length_unit.lower() == length:
                    return us

        logger.info("[Units] Tanınmayan CurrUnits=%r → N_MM", raw)
        return cls.N_MM

    @staticmethod
    def _normalize_v10(raw: str) -> str:
        """v10 `Ton. m. C` → `Tonf, m, C`."""
        s = raw.strip()
        if "." in s and "," not in s:
            s = ", ".join(p.strip() for p in s.split(".") if p.strip())
        tokens = [t.strip() for t in s.split(",")]
        out = []
        for t in tokens:
            out.append("tonf" if t.lower() == "ton" else t)
        return ", ".join(out)

    @property
    def force_unit(self) -> str:
        return self.value.split(",")[0].strip()

    @property
    def length_unit(self) -> str:
        return self.value.split(",")[1].strip()

    @property
    def temp_unit(self) -> str:
        return self.value.split(",")[2].strip()


# ============================================================================
# DÖNÜŞÜM FONKSİYONU
# ============================================================================
def factor(dim: Dimension, from_sys: UnitSystem, to_sys: UnitSystem) -> float:
    """Boyut ve iki sistem için dönüşüm faktörü (F^a · L^b modeli)."""
    if from_sys == to_sys:
        return 1.0

    F_f = FORCE_TO_N[from_sys.force_unit]
    F_t = FORCE_TO_N[to_sys.force_unit]
    L_f = LENGTH_TO_MM[from_sys.length_unit]
    L_t = LENGTH_TO_MM[to_sys.length_unit]

    if dim == Dimension.TEMP:
        a_f, _ = TEMP_FACTOR[from_sys.temp_unit]
        a_t, _ = TEMP_FACTOR[to_sys.temp_unit]
        return a_f / a_t
    if dim == Dimension.THERMAL_EXP:
        return 1.0

    force_ratio = F_f / F_t
    length_ratio = L_f / L_t

    if dim == Dimension.LENGTH or dim == Dimension.ACCEL:
        return length_ratio
    if dim == Dimension.FORCE:
        return force_ratio
    if dim == Dimension.STRESS or dim == Dimension.AREA_LOAD:
        return force_ratio / length_ratio**2
    if dim == Dimension.AREA:
        return length_ratio**2
    if dim == Dimension.I3:
        return length_ratio**3
    if dim == Dimension.I4:
        return length_ratio**4
    if dim == Dimension.I6:
        return length_ratio**6
    if dim == Dimension.MOMENT or dim == Dimension.MASS_INERTIA:
        return force_ratio * length_ratio
    if dim == Dimension.MASS or dim == Dimension.LINE_LOAD:
        return force_ratio / length_ratio
    if dim == Dimension.MASS_DENSITY:
        return force_ratio / length_ratio**4
    if dim == Dimension.WEIGHT_DENSITY:
        return force_ratio / length_ratio**3

    logger.info("[Units] Bilinmeyen Dimension: %r → factor=1.0", dim)
    return 1.0


def convert_value(dim, value, from_sys, to_sys):
    if dim is None or value is None or from_sys == to_sys:
        return value
    # ★ numpy array desteği
    if hasattr(value, "dtype") and hasattr(value, "shape"):
        try:
            import numpy as np
            return np.asarray(value, dtype=float) * factor(dim, from_sys, to_sys)
        except Exception:
            logger.info(
                "[Units] convert_value(%s, ndarray) → dönüştürülemedi",
                dim,
            )
            return value
    if isinstance(value, (tuple, list)):
        return type(value)(convert_value(dim, v, from_sys, to_sys) for v in value)
    try:
        return float(value) * factor(dim, from_sys, to_sys)
    except (TypeError, ValueError):
        logger.info("[Units] convert_value(%s, %r) → dönüştürülemedi", dim, value)
        return value


# ============================================================================
# BİRİM DEĞİŞİMİ EVENT
# ============================================================================
class UnitChangeEvent:
    def __init__(self):
        self._listeners: list[Callable] = []

    def subscribe(self, fn: Callable):
        self._listeners.append(fn)

    def emit(self, from_sys: UnitSystem, to_sys: UnitSystem, reason: str = ""):
        for fn in self._listeners:
            try:
                fn(from_sys, to_sys, reason)
            except Exception:
                logger.exception("UnitChangeEvent listener hatası")


unit_change_event = UnitChangeEvent()

def _convert_grid_lines(project, from_sys, to_sys) -> int:
    count = 0
    for gl in getattr(project, "grid_lines", []):
        if isinstance(gl, dict) and "ordinate" in gl and gl["ordinate"] is not None:
            gl["ordinate"] = convert_value(
                Dimension.LENGTH, gl["ordinate"], from_sys, to_sys,
            )
            count += 1
    return count

# ============================================================================
# PROJE DÖNÜŞÜMÜ
# ============================================================================
def ensure_nmm(project, *, reason: str = "analysis") -> int:
    return convert_project_units(project, UnitSystem.N_MM, reason=reason)

def convert_project_units(
    project, target: UnitSystem, *, reason: str = "",
) -> int:
    """Proje ağacındaki tüm birimli alanları hedef sisteme çevir."""
    from_sys = getattr(project, "unit_system", UnitSystem.N_MM)
    if from_sys == target:
        return 0

    n = _walk_convert(project, from_sys, target)
    n += _convert_grid_lines(project, from_sys, target)
    math_model = getattr(project, "math_model", None)
    if math_model is not None:
        for poly in getattr(math_model, "polygons", {}).values():
            if getattr(poly, "zone_children_by_direction", None):
                poly.zone_children_by_direction.clear()
                poly.show_zones_direction = None
    project.unit_system = target
    project.ui_dirty = True

    unit_change_event.emit(from_sys, target, reason)
    logger.info(
        "[Units] %s → %s (reason=%s, %d alan)",
        from_sys.value, target.value, reason, n,
    )
    return n


def _collect_schema(cls) -> dict[str, Dimension | None]:
    """MRO boyunca `__unit_schema__`'ları birleştir (alt sınıf kazanır)."""
    merged: dict[str, Dimension | None] = {}
    for c in reversed(cls.__mro__):
        sch = getattr(c, "__unit_schema__", None)
        if sch:
            merged.update(sch)
    return merged


def _walk_convert(root: Any, from_sys: UnitSystem, to_sys: UnitSystem) -> int:
    """Ağacı derinlemesine gez, __unit_schema__ olan her nesneyi dönüştür."""
    seen: set[int] = set()
    count = 0

    def visit(obj):
        nonlocal count
        if obj is None:
            return
        if isinstance(obj, (int, float, str, bool, bytes)):
            return
        oid = id(obj)
        if oid in seen:
            return
        seen.add(oid)

        schema = _collect_schema(type(obj))
        if schema:
            for field_path, dim in schema.items():
                count += _convert_field(obj, field_path, dim, from_sys, to_sys)

        if isinstance(obj, dict):
            for v in obj.values():
                visit(v)
        elif isinstance(obj, (list, tuple, set)):
            for v in obj:
                visit(v)
        elif hasattr(obj, "__dict__"):
            for v in list(obj.__dict__.values()):
                if v is obj:
                    continue
                visit(v)

    visit(root)
    return count


def _convert_field(
    obj, field_path: str, dim: Dimension | None,
    from_sys: UnitSystem, to_sys: UnitSystem,
) -> int:
    """obj.field_path (nested, a.b.c) alanını dönüştür.

    __dict__ doğrudan yaz → @property setter (mark_dirty) tetiklenmez.
    dict path'i desteklenir. Tuple/list değerler convert_value'da ele alınır.
    """
    if dim is None:
        return 0

    parts = field_path.split(".")
    target = obj
    for part in parts[:-1]:
        try:
            if isinstance(target, dict):
                target = target.get(part)
            elif hasattr(target, "__dict__"):
                target = target.__dict__.get(part)
            else:
                target = getattr(target, part, None)
        except Exception:
            return 0
        if target is None:
            return 0

    key = parts[-1]
    try:
        if isinstance(target, dict):
            if key not in target:
                return 0
            target[key] = convert_value(dim, target[key], from_sys, to_sys)
            return 1
        if hasattr(target, "__dict__") and key in target.__dict__:
            target.__dict__[key] = convert_value(
                dim, target.__dict__[key], from_sys, to_sys,
            )
            return 1
    except Exception as e:
        logger.info("[Units] %s dönüştürülemedi: %s", field_path, e)
    return 0


# ============================================================================
# DOĞRULAMA (opsiyonel)
# ============================================================================
def validate_project_units(project, original_system: UnitSystem) -> list[str]:
    """Roundtrip doğrulaması için stub."""
    return []

In [ ]:
# structload/windcalc/windengine.py

import math
from dataclasses import dataclass, field
from typing import Any

import numpy as np

from structload.data.wind_data import ARAZI_KATEGORILERI


@dataclass
class WindEngine:
    points: dict[str, tuple[float, float, float]]
    polygons: dict[str, list[str]]
    v_b0: float = 28.0
    terrain: str = "Kategori III"
    w_dir: Any = field(default_factory=lambda: np.array([1.0, 0.0, 0.0], dtype=float))
    scale_factor: float = 1000.0
    rho: float = 1.25
    ct: float = 1.0
    kI: float = 1.0

    # Calculated fields
    z0: float = field(init=False)
    zmin: float = field(init=False)
    geometry: dict[str, Any] = field(init=False, default_factory=dict)
    surfaces: dict[str, dict[str, Any]] = field(init=False, default_factory=dict)
    e: float | None = field(init=False, default=None)
    q_b: float = field(init=False, default=0.0)
    z_ref: float = field(init=False, default=0.0)
    kr: float = field(init=False, default=0.0)
    cr: float = field(init=False, default=0.0)
    Iv: float = field(init=False, default=0.0)
    ce: float = field(init=False, default=0.0)
    q_p: float = field(init=False, default=0.0)
    obb: dict[str, Any] = field(init=False, default_factory=dict)

    def __post_init__(self):
        # Ensure numpy array type
        self.w_dir = np.asarray(self.w_dir, dtype=float)

        if self.terrain not in ARAZI_KATEGORILERI:
            raise ValueError(f"Geçersiz arazi kategorisi: {self.terrain}")

        self.z0 = ARAZI_KATEGORILERI[self.terrain]["z0"]
        self.zmin = ARAZI_KATEGORILERI[self.terrain]["zmin"]

        # Complete unified geometry calculations
        self.update_geometry()
        self._calculate_wind_parameters()

    def update_geometry(self) -> None:
        """
        Bina geometrisini ve yönlendirilmiş sınır kutusunu (OBB) rüzgar yönüne göre
        tek bir ortak işlemde hesaplar ve sınıf özniteliklerini günceller.
        """
        if self.w_dir is None:
            return

        # 1. Rüzgar doğrultusunu ve dikini yerel 2B sisteme projekte etmek için baz vektörler
        w_xy = self.w_dir[:2] / np.linalg.norm(self.w_dir[:2])
        v_vec = np.array([-w_xy[1], w_xy[0]], dtype=float)

        pts_matrix = np.array(list(self.points.values()))

        # 2. Bina noktalarını bu rüzgar tabanlı eksenlere projekte etme
        u_vals = pts_matrix[:, :2] @ w_xy
        v_vals = pts_matrix[:, :2] @ v_vec
        z_vals = pts_matrix[:, 2]

        u_min, u_max = float(u_vals.min()), float(u_vals.max())
        v_min, v_max = float(v_vals.min()), float(v_vals.max())
        z_min, z_max = float(z_vals.min()), float(z_vals.max())

        # 3. Temel skaler boyutlar (Genişlik, Derinlik, Yükseklik)
        b = v_max - v_min   # Rüzgara dik cephe genişliği
        d = u_max - u_min   # Rüzgar yönündeki bina derinliği
        h = z_max - z_min   # Bina toplam yüksekliği
        e = min(b, 2.0 * h) # Kritik e uzunluğu

        # 4. Yönlendirilmiş sınır kutusu (OBB) köşe koordinatlarının 3D uzayda üretilmesi
        ref_point = np.array([0.0, 0.0, 0.0], dtype=float)

        def get_3d_coord(u: float, v: float) -> list[float]:
            return [
                ref_point[0] + w_xy[0] * u + v_vec[0] * v,
                ref_point[1] + w_xy[1] * u + v_vec[1] * v,
                0.0
            ]

        c1 = get_3d_coord(u_min, v_min)
        c2 = get_3d_coord(u_max, v_min)
        c3 = get_3d_coord(u_max, v_max)
        c4 = get_3d_coord(u_min, v_max)

        # 5. Rüzgarın çarptığı yüzeyin tam orta noktası (Impact Point)
        # Rüzgar yönü 'u' eksenidir. Çarpma noktası rüzgarın ilk çarptığı sınırda (u_min)
        # ve rüzgara dik cephenin tam ortasında (v'lerin ortalaması: v_mid) olmalıdır.
        v_mid = (v_min + v_max) / 2.0
        impact_point = get_3d_coord(u_min, v_mid)

        # Sınıf değişkenlerini güncelle
        self.e = e
        self.geometry = {
            "b": b,
            "d": d,
            "h": h,
            "z_ground": z_min,
            "z_ridge": z_max,
            "e": e
        }

        self.obb = {
            "obb_corners": [c1, c2, c3, c4],
            "impact_point": impact_point,
            "wind_dir": [w_xy[0], w_xy[1], 0.0],
            "parallel_dir": [v_vec[0], v_vec[1], 0.0]
        }

    def _calculate_wind_parameters(self) -> None:
        self.q_b = 0.5 * self.rho * (self.v_b0 ** 2) / 1000.0
        self.z_ref = max(self.geometry["z_ridge"], self.zmin)

        self.kr = 0.19 * ((self.z0 / 0.05) ** 0.07)
        self.cr = self.kr * math.log(self.z_ref / self.z0)
        self.Iv = self.kI / math.log(self.z_ref / self.z0)

        self.ce = (self.cr * self.ct) ** 2 * (1.0 + 7.0 * self.Iv)
        self.q_p = self.ce * self.q_b

    def report(self):
        report_= {
            "Geometri": dict(self.geometry),
            "Rüzgar": {
            "v_b0": self.v_b0,
            "terrain": self.terrain,
            "q_p": self.q_p,
            "z_ref": self.z_ref,
            "z0": self.z0,
            "zmin": self.zmin,
                "w_dir": self.w_dir
            }
        }
        return f'Rüzgar hesabı\n\n{report_}'

In [ ]:
from domain.definition import AreaUniformLoad, FrameDistributedLoad, LoadPattern

def build_wind_zone_fields(proj,
        surface: Polygon,
        direction_key: str,
        *,
        cpe_kind: str,
        cpi_pos: float,
        cpi_neg: float,
    ) -> list[tuple[list[tuple[float, float, float]], AreaUniformLoad]]:
    pressure_result = proj.loads.wind_results.get(direction_key)
    if pressure_result is None:
        raise ValueError(f"{direction_key} yönü için basınç sonucu yok.")

    q_p = float(getattr(pressure_result, "q_p", 0.0))
    if q_p <= 0.0:
        raise ValueError("Rüzgar tepe basıncı geçersiz.")

    wind_direction = np.asarray(
        getattr(pressure_result, "direction", getattr(pressure_result, "w_dir", (0.0, 0.0, 0.0))),
        dtype=float,
    )
    if wind_direction.shape != (3,) or not np.isfinite(wind_direction).all():
        raise ValueError("Rüzgar yön vektörü geçersiz.")

    zones = (surface.zone_children_by_direction or {}).get(direction_key, [])
    if not zones:
        raise ValueError(f"{direction_key} yönü için zone listesi boş.")

    surface_normal = np.asarray(Polygon.calculate_normal(list(surface.nodes)), dtype=float)
    sign = -1.0 if float(np.dot(surface_normal, wind_direction)) > 0.0 else 1.0
    kind_label = "CPE10" if cpe_kind == "cpe10" else "CPE1"
    override_key = "cpe10_override" if cpe_kind == "cpe10" else "cpe1_override"

    zone_fields: list[tuple[list[tuple[float, float, float]], AreaUniformLoad]] = []
    for z_dict in zones:
        raw_coords = z_dict.get("coords")
        if not raw_coords or len(raw_coords) < 3:
            continue
        coords = [(float(c[0]), float(c[1]), float(c[2])) for c in raw_coords]
        raw_coefs = z_dict.get(override_key) or z_dict.get(cpe_kind, (0.0, 0.0))
        if np.isscalar(raw_coefs):
            coefficients = (float(raw_coefs), float(raw_coefs))
        else:
            vals = tuple(float(v) for v in raw_coefs)
            coefficients = (vals[0], vals[1] if len(vals) > 1 else vals[0])

        for cpe_lbl, cpe in zip(("MN", "MX"), coefficients):
            for cpi_lbl, cpi in (("P", cpi_pos), ("M", cpi_neg)):
                pattern_name = f"WIND_{direction_key}_{kind_label}_{cpe_lbl}_{cpi_lbl}"
                pressure = sign * q_p * (cpe - cpi)
                value = float(
                    convert_value(
                        Dimension.AREA_LOAD,
                        pressure,
                        UnitSystem.KN_M,
                        proj.unit_system,
                    )
                )
                existing = proj.load_patterns.get(pattern_name)
                if existing is None:
                    proj.add_load_pattern(
                        LoadPattern(name=pattern_name, design_type="Wind", self_wt_mult=0.0)
                    )
                zone_fields.append(
                    (
                        coords,
                        AreaUniformLoad(
                            pattern_name=pattern_name,
                            direction=LoadDirection.LOCAL_3,
                            value=value,
                            source_name=f"{z_dict.get('label', '')} {kind_label} {cpe_lbl}/{cpi_lbl}",
                        ),
                    )
                )
    return zone_fields
    


In [ ]:
from domain.project import Project
path='D:\\Program\\cad4\\model3.cd4'
proj = Project.load(path)
poly= proj.polygons[299]

ad = proj.polygon_analysis_data(poly)
loads = list(ad.assigned_loads)

wind_direction_key= "01"
cpe_kind = "cpe10"
cpi_pos = 0.2
cpi_neg = -0.3

zone_fields = None
if wind_direction_key:
    zone_fields = build_wind_zone_fields(proj,
        poly,
        wind_direction_key,
        cpe_kind=cpe_kind,
        cpi_pos=cpi_pos,
        cpi_neg=cpi_neg,
    )
    
step = float(getattr(proj, "transfer_grid_step", 0.0) or 0.0)
n_bins = int(getattr(proj, "transfer_grid_bins", 12) or 12)
res_=build_polygon_frame_loads(
            proj.math_model,
            poly,
            loads=loads,
            analysis_data=ad,
            dist_type=DistType.from_input(ad.transfer_dist_type),
            step=step,
            n_bins=n_bins,
            zone_fields=zone_fields,
        )

In [ ]:
poly.associated

In [ ]:

def _extract_points_and_polygons(
        project,
        target_polygons: list[Any],
        to_wind_length: float,
    ) -> tuple[dict[str, list[float]], dict[str, list[str]]]:
        points: dict[str, list[float]] = {}
        polygons: dict[str, list[str]] = {}

        for polygon in target_polygons:
            if polygon.poly_type in {PolygonType.ZONE, PolygonType.SECTION_CUT}:
                continue

            vertices = getattr(polygon, "vertices", None) or getattr(polygon, "nodes", None)
            if not vertices or len(vertices) < 3:
                continue

            poly_name = polygon.label or f"POLYGON_{polygon.unique_id}"
            point_names: list[str] = []

            for idx, v in enumerate(vertices):
                if hasattr(v, "x") and hasattr(v, "y") and hasattr(v, "z"):
                    vx, vy, vz = float(v.x), float(v.y), float(v.z)
                    v_id = getattr(v, "unique_id", f"{polygon.unique_id}_{idx}")
                elif isinstance(v, (list, tuple)) and len(v) >= 3:
                    vx, vy, vz = float(v[0]), float(v[1]), float(v[2])
                    v_id = f"{polygon.unique_id}_{idx}"
                else:
                    continue

                point_name = f"V_{v_id}"
                points.setdefault(
                    point_name,
                    [
                        vx * to_wind_length,
                        vy * to_wind_length,
                        vz * to_wind_length,
                    ],
                )
                point_names.append(point_name)

            if len(point_names) >= 3:
                polygons[poly_name] = point_names

        return points, polygons
    
def _build_wind_config(project, proj_unit: UnitSystem) -> dict[str, Any]:
    cfg = project.config.wind
    terrain = str(getattr(cfg, "terrain_category", "III")).strip()
    if not terrain.lower().startswith("kategori"):
        terrain = f"Kategori {terrain}"
    return {
        "v_b0": float(getattr(cfg, "vb0", 28.0)),
        "terrain": terrain,
        "rho": float(getattr(cfg, "rho", 1.25)),
        "ct": float(getattr(cfg, "ct", 1.0)),
        "kI": float(getattr(cfg, "kI", 1.0)),
    }

def _length_scale(from_sys: UnitSystem, to_sys: UnitSystem) -> float:
    return float(convert_value(Dimension.LENGTH, 1.0, from_sys, to_sys))

from data.enums import PolygonType
explicit_surfaces = {
            PolygonType.SURFACE,
            PolygonType.LOAD_AREA,
        }

WIND_UNIT_SYSTEM = UnitSystem.KN_M
proj_unit = proj.unit_system

to_wind_length = _length_scale(proj_unit, WIND_UNIT_SYSTEM)

target_polygons = [
            p for p in proj.polygons.values()
            if p.poly_type in explicit_surfaces or not getattr(p, "is_temporary", False)
        ]
points, polygons = _extract_points_and_polygons(
            proj, target_polygons, to_wind_length,
        )

wind_config = _build_wind_config(proj, proj_unit)


w_dir= [1,0,0]

windengine = WindEngine(
                        points=points,
                        polygons=polygons,
                        w_dir=w_dir,
                        **wind_config,
                    )


In [ ]:
windengine.report()

In [ ]:
wind_config

In [ ]:
proj.config.wind

In [ ]:

import matplotlib.pyplot as plt

def plot_load_segments(segments):
    fig, ax = plt.subplots(figsize=(10, 4))

    for seg in segments:
        ax.plot(
            [seg.x1, seg.x2],
            [seg.q1, seg.q2],
            marker="o",
            linewidth=1.8
        )

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("x")
    ax.set_ylabel("q")
    ax.set_title("Load Distribution")
    ax.grid(True, linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()


In [ ]:
path='D:\\Program\\cad4\\model3.cd4'
proj = Project.load(path)
proj.load_patterns

In [ ]:
proj.math_model.frames[16].dist_loads[0]

In [15]:
from tools.s2k.loader import S2KLoader
s2k_path = r"D:\Projects\tests\simple_steel.$2k"
proj = S2KLoader(s2k_path).load()
print(proj.math_model.frames.keys())


dict_keys([105, 106, 107, 108, 109, 110, 111, 112])


In [16]:
load_fr=proj.math_model.frames[112].dist_loads
print(load_fr)

[FrameDistributedLoad(pattern_name='WIND', coord_sys='Local', load_dir='2', mode='REPLACE', force_or_moment='Force', direction=<LoadDirection.GLOBAL_Z: 'GZ'>, is_relative=True, coordinate_system='LOCAL', dist_type='RelDist', segments=[LoadSegment(x1=0.0, x2=0.25, q1=0.0, q2=2.0)]), FrameDistributedLoad(pattern_name='WIND', coord_sys='Local', load_dir='2', mode='REPLACE', force_or_moment='Force', direction=<LoadDirection.GLOBAL_Z: 'GZ'>, is_relative=True, coordinate_system='LOCAL', dist_type='RelDist', segments=[LoadSegment(x1=0.25, x2=0.75, q1=2.0, q2=1.5)]), FrameDistributedLoad(pattern_name='WIND', coord_sys='Local', load_dir='2', mode='REPLACE', force_or_moment='Force', direction=<LoadDirection.GLOBAL_Z: 'GZ'>, is_relative=True, coordinate_system='LOCAL', dist_type='RelDist', segments=[LoadSegment(x1=0.75, x2=1.0, q1=1.5, q2=1.0)]), FrameDistributedLoad(pattern_name='WIND', coord_sys='Local', load_dir='3', mode='REPLACE', force_or_moment='Force', direction=<LoadDirection.GLOBAL_Z: '